In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import classification_report

In [2]:
transaction = pd.read_csv("datasets/train_transaction.csv")

identity = pd.read_csv("datasets/train_identity.csv")

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

print(df.shape)

(590540, 434)


In [3]:
missing = (df.isnull().sum()/len(df))*100

drop_cols = missing[missing>80].index

df.drop(columns=drop_cols,inplace=True)

In [4]:
numeric_cols = df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [5]:
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [6]:
encoder = LabelEncoder()

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col].astype(str))

In [7]:
X = df.drop(
    columns=[
        "TransactionID",
        "isFraud"
    ]
)

y = df["isFraud"]

In [8]:
iso = IsolationForest(

    n_estimators=200,

    contamination=0.035,

    random_state=42,

    n_jobs=-1
)

iso.fit(X)

IsolationForest(contamination=0.035, n_estimators=200, n_jobs=-1,
                random_state=42)

In [10]:
pred = iso.predict(X)

In [14]:
pred = np.where(pred==-1,1,0)

In [16]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y, pred))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.97      1.00      0.98    569877
           1       0.00      0.00      0.00     20663

    accuracy                           0.97    590540
   macro avg       0.48      0.50      0.49    590540
weighted avg       0.93      0.97      0.95    590540



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
cm = confusion_matrix(y, pred)

print(cm)

[[569877      0]
 [ 20663      0]]


In [18]:
from sklearn.metrics import accuracy_score

print("Accuracy :", accuracy_score(y, pred))

Accuracy : 0.9650099908558268


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision :", precision_score(y, pred))
print("Recall    :", recall_score(y, pred))
print("F1 Score  :", f1_score(y, pred))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Precision : 0.0
Recall    : 0.0
F1 Score  : 0.0


In [20]:
import joblib

joblib.dump(iso, "isolation_model.pkl")

print("Isolation Forest model saved successfully!")

Isolation Forest model saved successfully!
